In [88]:
#!pip install nltk

In [89]:
#%pip install spacy

In [90]:
#import sys
#!{sys.executable} -m spacy download nl_core_news_lg

In [91]:
import pandas as pd
import nltk
import numpy as np

In [92]:
df = pd.read_csv("./dataset/cleaned_Leesmateriaal.csv")
SamenKolom = df["Gecombineerd"]

In [93]:
SamenKolom = SamenKolom.str.lower()

In [94]:
#cijfers met spatie ervoor en erna verwijderen
SamenKolom = SamenKolom.str.replace(r'(?<=\s)\d+(?=\s)', '', regex=True)

In [95]:
#puntkomma's en schuine strepen vervangen door spaties
SamenKolom = SamenKolom.str.replace(';', ' ')
SamenKolom = SamenKolom.str.replace('/', ' ')

In [96]:
#alle overige leestekens verwijderen
SamenKolom = SamenKolom.str.replace(r'[^\w\s]', '', regex=True)

In [97]:
#dubbele spaties verwijderen
SamenKolom = SamenKolom.str.replace(r'\s+', ' ', regex=True)
SamenKolom = SamenKolom.str.strip()

In [98]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('dutch'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vinvalentino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [99]:
#functie om stopwoorden te verwijderen
def verwijder_stopwoorden(zin):
    woorden = zin.split()
    nieuwe_woorden = []

    for woord in woorden:
        if woord.lower() not in stop_words:
            nieuwe_woorden.append(woord)

    return ' '.join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(verwijder_stopwoorden)

In [100]:
#functie om woorden te stammen (SnowballStemmer voor Nederlands)
# from nltk.stem.snowball import SnowballStemmer

# stemmer = SnowballStemmer("dutch")

# def stam_tekst(zin):
#     woorden = zin.split()
#     gestamde_woorden = []

#     for woord in woorden:
#         gestamde_woorden.append(stemmer.stem(woord))

#     return " ".join(gestamde_woorden)

# SamenKolom = SamenKolom.apply(stam_tekst)

In [101]:
#functie om woorden te lemmatiseren (Spacy voor Nederlands)
import spacy

nlp = spacy.load("nl_core_news_lg")

def lemmatiseer(zin):
    woorden = nlp(zin)
    nieuwe_woorden = []

    for woord in woorden:
        if not woord.is_punct:
            nieuwe_woorden.append(woord.lemma_)

    return " ".join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(lemmatiseer)

In [102]:
df["nlp"] = SamenKolom

In [103]:
df.to_csv("./dataset/cleaned_Leesmateriaal.csv", index=False)

In [104]:
#TF-IDF vectorizer importeren
from sklearn.feature_extraction.text import TfidfVectorizer

teksten = SamenKolom.fillna("").astype(str)

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(teksten)

print(X_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4435 stored elements and shape (283, 1082)>
  Coords	Values
  (0, 172)	0.2088331861016744
  (0, 425)	0.2088331861016744
  (0, 548)	0.1524003777216056
  (0, 874)	0.1602246107505877
  (0, 893)	0.14320092234151008
  (0, 1023)	0.1703118015291988
  (0, 632)	0.3892321784094844
  (0, 39)	0.2088331861016744
  (0, 698)	0.11757332005979104
  (0, 619)	0.1946160892047422
  (0, 310)	0.17670466539714896
  (0, 381)	0.1946160892047422
  (0, 72)	0.2088331861016744
  (0, 44)	0.1946160892047422
  (0, 363)	0.1703118015291988
  (0, 325)	0.1602246107505877
  (0, 173)	0.1703118015291988
  (0, 485)	0.11757332005979104
  (0, 960)	0.2088331861016744
  (0, 269)	0.04879036315148205
  (0, 1042)	0.4176663722033488
  (0, 2)	0.06527041779804334
  (0, 229)	0.085186025669289
  (0, 542)	0.08367225065293186
  (0, 692)	0.09262564414391751
  :	:
  (281, 495)	0.19966053760083052
  (281, 708)	0.20450049474583976
  (281, 636)	0.6481354566096464
  (281, 234)	0.20991

In [105]:
#TF-IDF resultaten omzetten naar DataFrame
import pandas as pd

tfidf_df = pd.DataFrame.sparse.from_spmatrix(
    X_tfidf,
    index=SamenKolom.index,
    columns=vectorizer.get_feature_names_out()
)

tfidf_df.head()

,2f,2f3f,3f,aandacht,aangrijpen,aankijken,aanleiding,aantrekkingkracht,aanwijzing,abba,...,zoektocht,zomers,zorg,zorgen,zoulfa,ztrk,zurhake,zutphen,zwaar,zwangerschap
0,NaN,NaN,0.06527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,0.074411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.082909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.188178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,0.07213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.081024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
SamenKolom[0:3]

0    dagboek joods meisje tijdens twee Wereldoorlog...
1    spannend thriller waarin vrouw confronteren st...
2    toegankelijk verhaal liefd relatie zoektocht p...
Name: Gecombineerd, dtype: str

In [107]:
from sklearn.metrics.pairwise import cosine_similarity

# Vergelijk alle leesmaterialen op basis van hun TF-IDF-vectoren.
cosine_sim = cosine_similarity(X_tfidf)
cataloguscodes = df["cataloguscode"].astype(str)
cosine_sim_df = pd.DataFrame(
    cosine_sim,
    index=cataloguscodes,
    columns=cataloguscodes
)

# Bekijk een klein deel van de matrix.
display(cosine_sim_df.iloc[:5, :5])

cataloguscode,CAT-00001,CAT-00002,CAT-00003,CAT-00004,CAT-00005
cataloguscode,,,,,
CAT-00001,1.000000,0.007571,0.003242,0.007339,0.021565
CAT-00002,0.007571,1.000000,0.020708,0.022172,0.060119
CAT-00003,0.003242,0.020708,1.000000,0.101556,0.041939
CAT-00004,0.007339,0.022172,0.101556,1.000000,0.074565
CAT-00005,0.021565,0.060119,0.041939,0.074565,1.000000


In [108]:
student_profiles = pd.read_csv("./dataset/cleaned_Student.csv").set_index("student_id")
materiaal_df = df.set_index("cataloguscode")

def aanbevolen_leesmateriaal(student_id, aantal=5):
    """Beveel beschikbare materialen aan die passen bij het profiel van een student."""
    student_id = str(student_id)
    if student_id not in student_profiles.index:
        raise KeyError(f"Onbekend student_id: {student_id}")
    if aantal < 1:
        raise ValueError("aantal moet minstens 1 zijn")

    profiel = student_profiles.loc[student_id]
    max_leestijd = pd.to_numeric(profiel["max_leestijd_min"], errors="coerce")
    if pd.isna(max_leestijd):
        raise ValueError(f"Geen geldige maximale leestijd voor student {student_id}")

    taal = str(profiel["voorkeur_taal"]).strip().casefold()
    beschikbare_boeken = (
        df["taal"].fillna("").astype(str).str.strip().str.casefold().eq(taal)
        & df["gegevensstatus"].fillna("").astype(str).str.strip().str.casefold().eq("beschikbaar")
        & pd.to_numeric(df["geschatte_leestijd_min"], errors="coerce").le(max_leestijd)
    )
    posities = beschikbare_boeken.to_numpy().nonzero()[0]
    if len(posities) == 0:
        raise ValueError(
            f"Geen beschikbare materialen gevonden voor taal {taal!r} "
            f"met een leestijd van maximaal {max_leestijd:g} minuten"
        )

    profiel_onderdelen = [
        profiel["interesses"],
        profiel["voorkeur_genre"],
        profiel["leesniveau"],
    ]
    profiel_tekst = " ".join(
        str(onderdeel).strip()
        for onderdeel in profiel_onderdelen
        if pd.notna(onderdeel)
        and str(onderdeel).strip().casefold() not in {"", "geen voorkeur"}
    )
    profiel_vector = vectorizer.transform([profiel_tekst])
    if profiel_vector.nnz == 0:
        raise ValueError(
            f"De profieltermen van student {student_id} komen niet voor in de boekteksten"
        )

    scores = cosine_similarity(profiel_vector, X_tfidf).ravel()[posities]
    resultaten = df.iloc[posities].copy()
    resultaten["profiel_similarity"] = scores

    niveau = str(profiel["leesniveau"]).strip().casefold()
    genre = str(profiel["voorkeur_genre"]).strip().casefold()
    materiaaltype = str(profiel["voorkeur_materiaaltype"]).strip().casefold()
    resultaten["zelfde_leesniveau"] = (
        resultaten["Niveau"].fillna("").astype(str).str.strip().str.casefold().eq(niveau)
    )
    resultaten["genre_match"] = (
        resultaten["genre"].fillna("").astype(str).str.strip().str.casefold().eq(genre)
        if genre not in {"", "geen voorkeur"}
        else False
    )
    resultaten["materiaaltype_match"] = (
        resultaten["Type materiaal"].fillna("").astype(str).str.strip().str.casefold().eq(materiaaltype)
        if materiaaltype not in {"", "gemengd", "geen voorkeur"}
        else False
    )

    resultaten = resultaten.sort_values(
        ["profiel_similarity", "zelfde_leesniveau", "genre_match", "materiaaltype_match"],
        ascending=False,
    )
    return resultaten[
        [
            "cataloguscode",
            "Titel",
            "Niveau",
            "genre",
            "Type materiaal",
            "geschatte_leestijd_min",
            "profiel_similarity",
        ]
    ].head(aantal)


def vergelijkbare_leesmaterialen(cataloguscode, aantal=5):
    """Geef de meest vergelijkbare materialen voor een cataloguscode."""
    cataloguscode = str(cataloguscode)
    if cataloguscode not in cosine_sim_df.index:
        raise KeyError(f"Onbekende cataloguscode: {cataloguscode}")
    if aantal < 1:
        raise ValueError("aantal moet minstens 1 zijn")

    scores = cosine_sim_df.loc[cataloguscode].drop(cataloguscode).nlargest(aantal)
    resultaten = materiaal_df.loc[scores.index, ["Titel"]].copy()
    resultaten["cosine_similarity"] = scores.values
    return resultaten

# Voorbeelden: profielaanbevelingen en vergelijkbare materialen.
display(aanbevolen_leesmateriaal("S001", aantal=5))
display(vergelijkbare_leesmaterialen("CAT-00001", aantal=5))

,cataloguscode,Titel,Niveau,genre,Type materiaal,geschatte_leestijd_min,profiel_similarity
179,CAT-00180,Hoe werkt DigiD?,2F,informatief,blogpost,16.0,0.282546
156,CAT-00157,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NaN,informatief,online artikel,7.0,0.266424
153,CAT-00154,Europese verboden op ongewenste AI-toepassinge...,3F+,informatief,online artikel,10.0,0.265767
269,CAT-00270,Schulden in Nederland: de stand van zaken,3F,informatief,online artikel,20.0,0.258140
152,CAT-00153,Nieuwe stap bij vernieuwing van curriculum,3F,informatief,online artikel,8.0,0.251567


,Titel,cosine_similarity
cataloguscode,,
CAT-00248,Floris,0.261229
CAT-00017,Haar leven,0.193790
CAT-00085,Het meisje dat er niet mocht zijn,0.163500
CAT-00046,De jongen in de gestreepte pyjama,0.149210
CAT-00042,Ik zal je vinden,0.125329


In [130]:
intdf = pd.read_csv("./dataset/cleaned_Interacties.csv")

intdf['intscore'] = intdf['beoordeling']

In [131]:
# Maak voor elke student-materiaalcombinatie een gemiddelde score.
pairs = (
    intdf.groupby(["student_id", "cataloguscode"], as_index=False)["intscore"]
    .mean()
)

k = 5
aantal_runs = 10
resultaten_per_run = []

for run in range(aantal_runs):
    # Gebruik per run een andere seed voor een nieuwe train/test-splitsing.
    test = pairs.groupby("student_id", group_keys=False).sample(
        frac=0.2,
        random_state=42 + run
    )

    # Een materiaal is relevant wanneer de testinteractie minimaal 4.0 scoort.
    relevante_items = (
        test.loc[test["intscore"] >= 4.0]
        .groupby("student_id")["cataloguscode"]
        .apply(set)
    )
    if relevante_items.empty:
        raise ValueError(
            f"Run {run + 1} bevat geen studenten met relevante testmaterialen."
        )

    evaluatieresultaten = []
    for student_id, relevante in relevante_items.items():
        try:
            aanbevelingen = (
                aanbevolen_leesmateriaal(student_id, aantal=k)["cataloguscode"]
                .tolist()
            )
        except (KeyError, ValueError):
            aanbevelingen = []

        hits = [item in relevante for item in aanbevelingen[:k]]

        # Precision@5 = aantal relevante items gedeeld door 5.
        precision_at_5 = sum(hits) / k

        # NDCG@5 = bonus voor relevante items die hoger in de lijst staan.
        dcg = sum(
            1 / np.log2(rank + 1)
            for rank, is_relevant in enumerate(hits, start=1)
            if is_relevant
        )
        ideale_aantal_hits = min(k, len(relevante))
        idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, ideale_aantal_hits + 1)
        )
        ndcg_at_5 = dcg / idcg if idcg else 0.0

        evaluatieresultaten.append({
            "student_id": student_id,
            "Precision@5": precision_at_5,
            "NDCG@5": ndcg_at_5,
        })

    evaluatie_scores = pd.DataFrame(evaluatieresultaten)
    resultaten_per_run.append({
        "Run": run + 1,
        "Precision@5": evaluatie_scores["Precision@5"].mean(),
        "NDCG@5": evaluatie_scores["NDCG@5"].mean(),
    })

run_scores = pd.DataFrame(resultaten_per_run)
print("Scores per run:")
display(run_scores)

print("Gemiddelde en standaardafwijking over 10 runs:")
display(run_scores[["Precision@5", "NDCG@5"]].agg(["mean", "std"]))

Gemiddelde Precision@5: 0.018918918918918916
Gemiddelde NDCG@5: 0.04375233065016954


,student_id,Precision@5,NDCG@5
0,S001,0.0,0.0
1,S002,0.0,0.0
2,S003,0.0,0.0
3,S005,0.0,0.0
4,S006,0.0,0.0
...,...,...,...
69,S095,0.0,0.0
70,S097,0.0,0.0
71,S098,0.0,0.0
72,S099,0.0,0.0


In [111]:

# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", intdf.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": intdf.nunique(),
    "Aantal missende waarden": intdf.isna().sum(),
    "Datatype": intdf.dtypes,
    "Aantal dubbele waarden": intdf.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (2000, 9)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
interactie_id,2000,0,str,0
student_id,100,0,str,1900
cataloguscode,276,0,str,1724
interactietype,6,0,str,1994
beoordeling,5,373,float64,1622
leespercentage,101,0,int64,1899
leestijd_min,560,0,int64,1440
datum,288,0,str,1712
intscore,23,0,float64,1977


In [112]:
intdf[1995:2000]

,interactie_id,student_id,cataloguscode,interactietype,beoordeling,leespercentage,leestijd_min,datum,intscore
1995,I01996,S100,CAT-00030,gelezen,4.0,99,261,2026-04-30,3.8
1996,I01997,S100,CAT-00250,niet_afgemaakt,3.0,60,98,2026-07-16,2.3
1997,I01998,S100,CAT-00066,gelezen,4.0,96,698,2026-09-03,3.8
1998,I01999,S100,CAT-00009,bekeken,NaN,57,318,2026-01-08,2.3
1999,I02000,S100,CAT-00143,bekeken,3.0,37,3,2026-10-15,2.5


In [113]:
intdf["interactietype"].unique().tolist()

['opgeslagen', 'bekeken', 'gelezen', 'niet_afgemaakt', 'geliket', 'afgewezen']

In [114]:
inttype = intdf["interactietype"]

In [115]:
inttype.unique().tolist()

['opgeslagen', 'bekeken', 'gelezen', 'niet_afgemaakt', 'geliket', 'afgewezen']

In [116]:
#interactietypes waardes geven van 1 tot 5
inttype.loc[inttype == "geliket"] = "5"
inttype.loc[inttype == "opgeslagen"] = "4"
inttype.loc[inttype == "gelezen"] = "3"
inttype.loc[inttype == "bekeken"] = "2"
inttype.loc[inttype == "niet_afgemaakt"] = "1"
inttype.loc[inttype == "afgewezen"] = "1"
inttype = inttype.astype(float)

In [117]:
leesprc = intdf["leespercentage"]

In [118]:
#leespercentage waardes geven van 1 tot 5
leesprc = leesprc.apply(lambda x: round(x / 20))
leesprc = leesprc.astype(float)

In [119]:
leesprc.unique().tolist()

[4.0, 2.0, 1.0, 5.0, 0.0, 3.0]

In [120]:
#interacties score berekenen, lege waardes worden niet meegenomen
scores = pd.DataFrame({
    "leesprc": leesprc,
    "inttype": inttype,
    "beoordeling": intdf["beoordeling"],
}, index=intdf.index)

weights = pd.Series({
    "leesprc": 0.15,
    "inttype": 0.35,
    "beoordeling": 0.50,
})

weighted_sum = scores.mul(weights, axis="columns").sum(axis=1, min_count=1)
available_weight = scores.notna().mul(weights, axis="columns").sum(axis=1)

intscore = weighted_sum / available_weight

In [121]:
intscore

0       4.00
1       2.50
2       4.00
3       3.65
4       1.50
        ... 
1995    3.80
1996    2.30
1997    3.80
1998    2.30
1999    2.50
Length: 2000, dtype: float64

In [122]:
intdf["intscore"] = intscore

In [123]:
intdf.head()

,interactie_id,student_id,cataloguscode,interactietype,beoordeling,leespercentage,leestijd_min,datum,intscore
0,I00001,S001,CAT-00223,opgeslagen,4.0,88,305,2026-01-17,4.00
1,I00002,S001,CAT-00253,bekeken,3.0,42,93,2026-08-24,2.50
2,I00003,S001,CAT-00055,opgeslagen,4.0,84,299,2026-10-14,4.00
3,I00004,S001,CAT-00266,gelezen,4.0,82,235,2026-05-04,3.65
4,I00005,S001,CAT-00262,niet_afgemaakt,2.0,14,2,2026-02-19,1.50


In [124]:
intdf.to_csv("./dataset/cleaned_Interacties.csv", index=False)

In [125]:
intdf["intscore"] = intscore

student_item_matrix = intdf.pivot_table(
    index="student_id",
    columns="cataloguscode",
    values="intscore",
    aggfunc="mean"
)

student_item_matrix.head()

cataloguscode,CAT-00001,CAT-00002,CAT-00003,CAT-00004,CAT-00005,CAT-00006,CAT-00007,CAT-00008,CAT-00009,CAT-00010,...,CAT-00274,CAT-00275,CAT-00276,CAT-00277,CAT-00278,CAT-00279,CAT-00280,CAT-00281,CAT-00282,CAT-00283
student_id,,,,,,,,,,,,,,,,,,,,,
S001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.65,NaN,NaN
S002,NaN,NaN,NaN,NaN,4.15,NaN,NaN,NaN,2.15,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
S003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
S004,1.65,NaN,NaN,NaN,NaN,NaN,2.8,NaN,2.50,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN
S005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [126]:
student_item_matrix.to_csv("./matrix/student_item_matrix.csv", index=True)

In [127]:
# Vergelijk de scorepatronen van alle studenten op materialen.
# .T draait de matrix om, zodat corr() de studenten met elkaar vergelijkt.
# min_periods=3 vereist minstens 3 materialen met scores van beide studenten.
student_similarity = student_item_matrix.T.corr(min_periods=3)

# Maak een matrix waarin 1 betekent dat een score bekend is en 0 dat die ontbreekt.
# De matrixvermenigvuldiging telt per studentenpaar hoeveel materialen zij delen.
overlap = (
    student_item_matrix.notna().astype(int)
    @ student_item_matrix.notna().astype(int).T
)

# Kies voor dit voorbeeld de eerste student uit de matrix.
# De volgende cel gebruikt expliciet S001.
student_id = student_similarity.index[0]
# Pak de correlaties van deze student met alle andere studenten.
neighbors = (
    student_similarity[student_id]
    # Verwijder de student zelf: die is altijd volledig gelijk aan zichzelf.
    .drop(index=student_id)
    # Verwijder studenten waarvoor geen correlatie berekend kon worden.
    .dropna()
    # Zet de hoogste correlaties bovenaan.
    .sort_values(ascending=False)
    # Bewaar de 5 hoogste correlaties (die kunnen ook negatief zijn).
    .head(5)
)
# Toon de meest vergelijkbare studenten en het aantal gedeelde materialen
# Toon de gekozen student en de correlaties met diens vergelijkbare studenten.
print("Student:", student_id)
print("Vergelijkbare studenten:")
print(neighbors)
# Toon hoeveel materialen deze studenten met de gekozen student gemeen hebben.
print("Gedeelde materialen:")
print(overlap.loc[student_id, neighbors.index])

Student: S001
Vergelijkbare studenten:
student_id
S051    0.964056
S038    0.911373
S002    0.900643
S006    0.805943
S014    0.790474
Name: S001, dtype: float64
Gedeelde materialen:
student_id
S051    3
S038    4
S002    3
S006    3
S014    3
Name: S001, dtype: int64


In [128]:
# Kies de student voor wie we aanbevelingen willen maken.
target = "S001"

# Haal de correlaties van deze student met anderen op.
# Verwijder de student zelf en correlaties die niet berekend konden worden.
neighbors = student_similarity[target].drop(target).dropna()
# Houd alleen positief vergelijkbare studenten over en kies de 5 sterkste buren.
neighbors = neighbors[neighbors > 0].nlargest(5)

# Haal de scores op die deze buren aan alle materialen hebben gegeven.
neighbor_scores = student_item_matrix.loc[neighbors.index]

# Vermenigvuldig elke score met de gelijkenis van de buur en tel dit per materiaal op.
# min_count=1 zorgt dat een materiaal zonder scores NaN blijft in plaats van 0.
weighted_sum = neighbor_scores.mul(neighbors, axis=0).sum(axis=0, min_count=1)
# Tel per materiaal de gelijkenissen op van de buren die het hebben beoordeeld.
weight_sum = neighbor_scores.notna().mul(neighbors, axis=0).sum(axis=0)
# Deel de gewogen scoresom door de gebruikte gewichten voor een voorspelde score.
predicted_scores = weighted_sum / weight_sum

# Tel hoeveel buren voor elk materiaal een score hebben gegeven.
support = neighbor_scores.notna().sum(axis=0)
# Bepaal welke materialen de gekozen student nog niet heeft beoordeeld.
unseen = student_item_matrix.loc[target].isna()

# Houd alleen onbekende materialen over die door minstens 2 buren zijn beoordeeld.
# Sorteer op voorspelde score en selecteer de 10 hoogste aanbevelingen.
recommendations = (
    predicted_scores[unseen & (support >= 2)]
    .sort_values(ascending=False)
    .head(10)
)

# Toon de aanbevolen cataloguscodes met hun voorspelde scores.
print(recommendations)

cataloguscode
CAT-00156    4.318459
CAT-00181    3.830951
CAT-00128    3.727551
CAT-00264    3.650000
CAT-00034    3.269304
CAT-00030    3.058847
CAT-00195    3.041780
CAT-00079    3.028745
CAT-00202    2.851141
CAT-00157    2.806359
dtype: float64


In [129]:
# Eén score per student-materiaalcombinatie; voorkom dat dezelfde combinatie
# zowel in train als test terechtkomt.
pairs = (
    intdf.groupby(["student_id", "cataloguscode"], as_index=False)["intscore"]
    .mean()
)

# Houd per student willekeurig 20% van de interacties achter.
test = pairs.groupby("student_id", group_keys=False).sample(
    frac=0.2, random_state=42
)
train = pairs.drop(index=test.index)

# Bouw de matrix opnieuw op met uitsluitend train-data.
students = pairs["student_id"].unique()
items = pairs["cataloguscode"].unique()

train_matrix = (
    train.pivot(index="student_id", columns="cataloguscode", values="intscore")
    .reindex(index=students, columns=items)
)

# Bereken opnieuw de student-similarity op train-data.
similarity = train_matrix.T.corr(min_periods=3)


def recommend(user_id, k=3):
    neighbors = similarity.loc[user_id].drop(index=user_id).dropna()
    neighbors = neighbors[neighbors > 0].nlargest(k)

    if neighbors.empty:
        return []

    neighbor_scores = train_matrix.loc[neighbors.index]
    weighted_sum = neighbor_scores.mul(neighbors, axis=0).sum(axis=0, min_count=1)
    weight_sum = neighbor_scores.notna().mul(neighbors, axis=0).sum(axis=0)
    predicted = weighted_sum / weight_sum

    support = neighbor_scores.notna().sum(axis=0)
    unseen = train_matrix.loc[user_id].isna()

    return predicted[unseen & (support >= 2)].nlargest(k).index.tolist()


# Definieer vooraf wat je als een relevante interactie ziet.
# 4.0 is een startpunt op jouw intscore-schaal; pas dit aan als dat beter
# aansluit bij jouw definitie van "interessant".
relevant_items = (
    test.loc[test["intscore"] >= 4.0]
    .groupby("student_id")["cataloguscode"]
    .apply(set)
)

results = []
k = 5

for student_id, relevant in relevant_items.items():
    try:
        recommendations = recommend(student_id, k=k)
    except (KeyError, ValueError):
        recommendations = []

    hits = [item in relevant for item in recommendations[:k]]

    # Precision@5: aandeel relevante materialen in de top 5.
    precision_at_5 = sum(hits) / k

    # NDCG@5: relevante materialen hoger in de lijst krijgen meer gewicht.
    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, is_relevant in enumerate(hits, start=1)
        if is_relevant
    )
    ideal_hits = min(k, len(relevant))
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )
    ndcg_at_5 = dcg / idcg if idcg else 0.0

    results.append({
        "student_id": student_id,
        "Precision@5": precision_at_5,
        "NDCG@5": ndcg_at_5,
    })

scores = pd.DataFrame(results)
print(scores[["Precision@5", "NDCG@5"]].mean())
display(scores)

Precision@5    0.017391
NDCG@5         0.042148
dtype: float64


,student_id,Precision@5,NDCG@5
0,S003,0.0,0.000000
1,S010,0.0,0.000000
2,S012,0.0,0.000000
3,S013,0.0,0.000000
4,S017,0.0,0.000000
5,S020,0.0,0.000000
6,S022,0.0,0.000000
7,S024,0.0,0.000000
8,S030,0.0,0.000000
9,S032,0.2,0.630930
